# Level 1- Task 1: Data Collection and Web Scraping

- Collecting Exchange Rate Data from the Central Bank of Nigeria

- Project Objective

This project aims to collect exchange rate data published by the Central Bank of Nigeria (CBN) and store it in a structured format for further analysis.

Using Python, I will inspect the webpage, identify the relevant data, extract the available records, and save the results as a CSV file. This will provide a reusable dataset for subsequent data cleaning and exploratory data analysis.

- Inspecting the Website Response

Before extracting the exchange-rate records, I will inspect the webpage response to determine whether the table is available in the HTML returned by the website. This will help establish the appropriate method for collecting the data.

In [1]:
import requests
from bs4 import BeautifulSoup

## Requesting the Webpage

I will send a request to the CBN exchange-rates page and inspect the response. This will help determine whether the page can be retrieved successfully and whether its HTML contains the data we need.

In [2]:
url = "https://www.cbn.gov.ng/rates/"

response = requests.get(url, timeout=30)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response length:", len(response.text))
print(response.text[:500])

Status code: 200
Content type: text/html
Response length: 16263
<!DOCTYPE html>
<html dir="ltr" lang="en-US">
	<script>
		function includeHTML() {
		  var z, i, elmnt, file, xhttp;
		  /*loop through a collection of all HTML elements:*/
		  z = document.getElementsByTagName("*");
		  for (i = 0; i < z.length; i++) {
			elmnt = z[i];
			/*search for elements with a certain atrribute:*/
			file = elmnt.getAttribute("w3-include-html");
			if (file) {
			  /*make an HTTP request using the attribute value as the file name:*/
			  xhttp = new XMLHttpR


## Checking the HTML Structure

I will examine the returned HTML to identify any tables and check whether exchange-rate information is present. This will help determine whether BeautifulSoup can extract the records directly or whether the data must be retrieved separately.

In [3]:
soup = BeautifulSoup(response.text, "html.parser")

print("Page title:", soup.title.get_text(strip=True) if soup.title else "No title found")
print("Number of HTML tables:", len(soup.find_all("table")))

for i, table in enumerate(soup.find_all("table"), start=1):
    print(f"\nTable {i}:")
    print(table.get_text(" ", strip=True)[:500])

print("\nContains 'NFEM':", "NFEM" in response.text)
print("Contains 'Exchange Rates':", "Exchange Rates" in response.text)

Page title: Data & Statistics | Central Bank of Nigeria
Number of HTML tables: 0

Contains 'NFEM': False
Contains 'Exchange Rates': True


## Identifying Data Sources

The initial HTML response contains no table elements, and the exchange-rate identifier we checked is absent. I will inspect the page's script references and relevant links to investigate how the exchange-rate data is loaded.

In [4]:
# Find JavaScript files referenced by the page
scripts = soup.find_all("script", src=True)

print("JavaScript files referenced:", len(scripts))

for script in scripts:
    print(script["src"])

# Find links whose text or URL may relate to exchange rates or data
print("\nRelevant links:")

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    href = link["href"]

    if any(word in (text + " " + href).lower()
           for word in ["rate", "exchange", "data", "statistic", "api"]):
        print(f"Text: {text!r} | URL: {href}")

JavaScript files referenced: 6
https://www.googletagmanager.com/gtag/js?id=G-N4BDKFWBKH
/js/jquery.js
/js/plugins.min.js
/js/components/bs-select.js
/js/components/selectsplitter.js
/js/functions.js

Relevant links:
Text: 'Crude Oil Price' | URL: /rates/crudeoil.html
Text: 'Daily Crude Oil Price' | URL: /rates/DailyCrude.html
Text: 'Discount Windows' | URL: /rates/DiscWindow.html
Text: 'Economic/Macro Indicators' | URL: /rates/MacroIndicators.html
Text: 'Exchange Rates' | URL: /rates/ExchRateByCurrency.html
Text: 'Financial Data' | URL: /rates/FinancialData.html
Text: 'Government Securities Details' | URL: /rates/GovtSecuritiesDrillDown.html
Text: 'Government Securities Summary' | URL: /rates/GovtSecurities.html
Text: 'Gross Domestic Product \xa0- Nominal' | URL: /rates/NominalGDP.html
Text: 'Gross Domestic Product \xa0- Real' | URL: /rates/RealGDP.html
Text: 'Inflation' | URL: /rates/inflrates.html
Text: 'Inter-bank Rates' | URL: /rates/interbankrates.html
Text: 'Money and Credit Stat

## Inspecting the Exchange Rate Data Page

The CBN statistics page links to a dedicated Exchange Rates page. I will inspect this page directly to determine whether its HTML contains the exchange-rate records we need to collect.

In [5]:
exchange_url = "https://www.cbn.gov.ng/rates/ExchRateByCurrency.html"

exchange_response = requests.get(exchange_url, timeout=30)

print("Status code:", exchange_response.status_code)
print("Page title:", BeautifulSoup(
    exchange_response.text, "html.parser"
).title.get_text(strip=True))

exchange_soup = BeautifulSoup(exchange_response.text, "html.parser")

tables = exchange_soup.find_all("table")
print("Number of HTML tables:", len(tables))

for i, table in enumerate(tables, start=1):
    print(f"\nTable {i}:")
    print(table.get_text(" ", strip=True)[:700])

Status code: 200
Page title: Exchange Rates | Central Bank of Nigeria
Number of HTML tables: 1

Table 1:
NIGERIAN FOREIGN EXCHANGE MARKET (NFEM) RATES (â¦/US$) Date NFEM Rate (â¦/US$) Highest Rate (â¦/US$) Lowest Rate (â¦/US$) Closing Rate  (â¦/US$) Simple Aver. Rate (Mean) (â¦/US$) NFEM Interbank Turnover (US$) No. of Deals at Interbank NFEM Total Turnover (US$) No. of Deals at NFEM


## Examining the Exchange Rate Table

The dedicated CBN page contains an HTML table for Nigerian Foreign Exchange Market (NFEM) rates. I will inspect its column headings and sample records to understand the available fields before extracting the data.

In [6]:
table = exchange_soup.find("table")

headers = [
    cell.get_text(" ", strip=True)
    for cell in table.find_all("th")
]

rows = table.find_all("tr")

print("Column headings:")
print(headers)

print("\nNumber of table rows:", len(rows))

print("\nFirst five rows:")

for row in rows[:5]:
    values = [
        cell.get_text(" ", strip=True)
        for cell in row.find_all(["th", "td"])
    ]
    print(values)

Column headings:
[]

Number of table rows: 2

First five rows:
['NIGERIAN FOREIGN EXCHANGE MARKET (NFEM) RATES (â\x82¦/US$)']
['Date', 'NFEM Rate (â\x82¦/US$)', 'Highest Rate (â\x82¦/US$)', 'Lowest Rate (â\x82¦/US$)', 'Closing Rate  (â\x82¦/US$)', 'Simple Aver. Rate (Mean) (â\x82¦/US$)', 'NFEM Interbank Turnover (US$)', 'No. of Deals at Interbank', 'NFEM Total Turnover (US$)', 'No. of Deals at NFEM']


## Extracting the Exchange Rate Records

I will extract the text from each row and its cells to organise the exchange-rate records into a structured dataset. This will allow the collected data to be checked, cleaned, and analysed in subsequent tasks.

In [7]:
import pandas as pd

## Converting the Table into a DataFrame

I will extract the rows and columns from the selected HTML table and organise them into a pandas DataFrame. This will make the collected records easier to inspect and reuse for further analysis.

In [8]:
print("Number of headers:", len(headers))
print("Headers:", headers)

first_row = table.find("tr")
print("\nFirst row HTML:")
print(first_row.prettify()[:2000] if first_row else "No row found")

Number of headers: 0
Headers: []

First row HTML:
<tr>
 <td colspan="10">
  <b>
   NIGERIAN FOREIGN EXCHANGE MARKET (NFEM) RATES (â¦/US$)
  </b>
 </td>
</tr>



In [9]:
for i, row in enumerate(table.find_all("tr")[:6], start=1):
    cells = row.find_all(["th", "td"])
    values = [cell.get_text(" ", strip=True) for cell in cells]

    print(f"Row {i}: {values}")

Row 1: ['NIGERIAN FOREIGN EXCHANGE MARKET (NFEM) RATES (â\x82¦/US$)']
Row 2: ['Date', 'NFEM Rate (â\x82¦/US$)', 'Highest Rate (â\x82¦/US$)', 'Lowest Rate (â\x82¦/US$)', 'Closing Rate  (â\x82¦/US$)', 'Simple Aver. Rate (Mean) (â\x82¦/US$)', 'NFEM Interbank Turnover (US$)', 'No. of Deals at Interbank', 'NFEM Total Turnover (US$)', 'No. of Deals at NFEM']


## Extracting the Exchange Rate Records

The initial inspection revealed that the CBN table uses a title row followed by a separate row containing the column names. The column names are stored in ordinary table cells rather than header cells.

To address this, I will extract the column names from the second row and collect the data records from the rows that follow. The extracted values will then be organised into a pandas DataFrame for validation and further use.

In [10]:
rows = table.find_all("tr")

headers = [
    cell.get_text(" ", strip=True)
    for cell in rows[1].find_all(["th", "td"])
]

data_rows = []

for row in rows[2:]:
    cells = row.find_all(["th", "td"])
    values = [cell.get_text(" ", strip=True) for cell in cells]

    if len(values) == len(headers):
        data_rows.append(values)

df = pd.DataFrame(data_rows, columns=headers)

print("Column headings:", headers)
print("Dataset shape:", df.shape)
display(df.head())

Column headings: ['Date', 'NFEM Rate (â\x82¦/US$)', 'Highest Rate (â\x82¦/US$)', 'Lowest Rate (â\x82¦/US$)', 'Closing Rate  (â\x82¦/US$)', 'Simple Aver. Rate (Mean) (â\x82¦/US$)', 'NFEM Interbank Turnover (US$)', 'No. of Deals at Interbank', 'NFEM Total Turnover (US$)', 'No. of Deals at NFEM']
Dataset shape: (0, 10)


,Date,NFEM Rate (â¦/US$),Highest Rate (â¦/US$),Lowest Rate (â¦/US$),Closing Rate (â¦/US$),Simple Aver. Rate (Mean) (â¦/US$),NFEM Interbank Turnover (US$),No. of Deals at Interbank,NFEM Total Turnover (US$),No. of Deals at NFEM


## Investigating References to the Data Source

The initial HTML response contained the exchange-rate table title and ten column headings, but no actual data records. This meant that extracting the table directly from the downloaded HTML was not sufficient.

To investigate how the webpage obtains its data, I will search the HTML response for URLs and references containing the term api. 

In [11]:
import re

api_matches = re.findall(
    r"""https?://[^"' ]+|[^"' ]*api[^"' ]*""",
    exchange_response.text,
    flags=re.IGNORECASE
)

for match in api_matches[:20]:
    print(match)

https://www.clarity.ms/tag/
https://www.googletagmanager.com/gtag/js?id=G-N4BDKFWBKH
https://kendo.cdn.telerik.com/themes/7.2.1/default/default-ocean-blue.css
https://kendo.cdn.telerik.com/themes/7.2.0/default/default-main.css
https://code.jquery.com/jquery-1.12.3.min.js
https://kendo.cdn.telerik.com/2024.1.319/js/kendo.all.min.js
https://unpkg.com/jszip/dist/jszip.min.js
https://cdnjs.cloudflare.com/ajax/libs/xlsx/0.17.0/xlsx.full.min.js
https://maxcdn.bootstrapcdn.com/bootstrap/4.5.2/css/bootstrap.min.css
https://fonts.googleapis.com/css?family=Lato:300,400,400i,700|Poppins:300,400,500,600,700|PT+Serif:400,400i&display=swap
/api/GetAllNFEM_RatesGRAPH
/api/GetAllExchangeRatesGRAPH
/api/GetAllExchangeRates


## Testing the Exchange Rate Endpoint

The HTML investigation revealed three endpoint paths: GetAllNFEM_RatesGRAPH, GetAllExchangeRatesGRAPH, and GetAllExchangeRates. The initial table extraction returned an empty DataFrame with ten columns and zero records, so the underlying exchange-rate data still needs to be retrieved.

I will first test GetAllExchangeRates because its name suggests that it may return exchange-rate records rather than data intended specifically for a graph. The response will be inspected to determine whether the endpoint is accessible and what format of data it returns.

We chose GetAllExchangeRates instead of the NFEM endpoints because it covers multiple currencies and includes buying, central, and selling rates, giving us a richer dataset for web scraping and data cleaning than the NFEM endpoint, which appears to focuses on daily USD/naira market indicators.

In [12]:
from urllib.parse import urljoin

api_url = urljoin(
    "https://www.cbn.gov.ng/Rates/ExchRateByCurrency.html",
    "/api/GetAllExchangeRates"
)

api_response = requests.get(api_url, timeout=30)

print("Status code:", api_response.status_code)
print("Content type:", api_response.headers.get("Content-Type"))
print("Response preview:")
print(api_response.text[:1000])

Status code: 200
Content type: application/json; charset=utf-8
Response preview:
[{"id":64143,"currency":"CFA","ratedate":"2026-10-09","buyingrate":"2.2508","centralrate":"2.2608","sellingrate":"2.2708"},{"id":64144,"currency":"YUAN/RENMINBI","ratedate":"2026-10-09","buyingrate":"198.7637","centralrate":"198.8384","sellingrate":"198.9131"},{"id":64145,"currency":"DANISH KRONA","ratedate":"2026-10-09","buyingrate":"199.2639","centralrate":"199.3388","sellingrate":"199.4137"},{"id":64146,"currency":"EURO","ratedate":"2026-10-09","buyingrate":"1489.6753","centralrate":"1490.2353","sellingrate":"1490.7952"},{"id":64147,"currency":"YEN","ratedate":"2026-10-09","buyingrate":"8.4024","centralrate":"8.4056","sellingrate":"8.4087"},{"id":64148,"currency":"RIYAL","ratedate":"2026-10-09","buyingrate":"354.2439","centralrate":"354.3771","sellingrate":"354.5103"},{"id":64149,"currency":"SOUTH AFRICAN RAND","ratedate":"2026-10-09","buyingrate":"80.4136","centralrate":"80.4438","sellingrate":"80.4741

## Inspecting the Exchange Rate Data Structure

The endpoint returned a successful response with a JSON content type. Its response preview contains exchange-rate fields and values, including currency, buying rate, central rate, selling rate, and date.

Before converting the response into a DataFrame, I will inspect its JSON structure to determine whether the records are stored as a list of dictionaries or nested inside another object. This will help identify the correct way to extract the records without losing any fields.

In [13]:
api_data = api_response.json()

print("Response type:", type(api_data).__name__)

if isinstance(api_data, dict):
    print("Top-level keys:", list(api_data.keys()))

elif isinstance(api_data, list):
    print("Number of records:", len(api_data))
    print("First record:", api_data[0] if api_data else "No records found")

Response type: list
Number of records: 62333
First record: {'id': 64143, 'currency': 'CFA', 'ratedate': '2026-10-09', 'buyingrate': '2.2508', 'centralrate': '2.2608', 'sellingrate': '2.2708'}


## Converting the API Response into a DataFrame

The API returned 62,333 records as a list of dictionaries. Each dictionary contains exchange-rate information, including the currency, date, and buying, central, and selling rates.

Since the records already have named fields, I can convert them directly into a pandas DataFrame. This will organise the data into rows and columns so I can inspect the records, check data quality, and prepare the dataset for further analysis.

In [14]:
df = pd.DataFrame(api_data)

print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())

display(df.head(10))

Dataset shape: (62333, 6)
Column names: ['id', 'currency', 'ratedate', 'buyingrate', 'centralrate', 'sellingrate']


,id,currency,ratedate,buyingrate,centralrate,sellingrate
0,64143,CFA,2026-10-09,2.2508,2.2608,2.2708
1,64144,YUAN/RENMINBI,2026-10-09,198.7637,198.8384,198.9131
2,64145,DANISH KRONA,2026-10-09,199.2639,199.3388,199.4137
3,64146,EURO,2026-10-09,1489.6753,1490.2353,1490.7952
4,64147,YEN,2026-10-09,8.4024,8.4056,8.4087
5,64148,RIYAL,2026-10-09,354.2439,354.3771,354.5103
6,64149,SOUTH AFRICAN RAND,2026-10-09,80.4136,80.4438,80.4741
7,64150,SDR,2026-10-09,1800.0875,1800.7641,1801.4408
8,64151,SWISS FRANC,2026-10-09,1599.1657,1599.7668,1600.3679
9,64152,POUNDS STERLING,2026-10-09,1757.8408,1758.5015,1759.1623


## Testing API Pagination

The exchange-rate endpoint returned 62,333 records in the current extraction. Before saving the dataset as the final raw collection, I will test whether the endpoint responds differently when a page parameter is supplied.

This test will help determine whether the API may support pagination. A single test is not sufficient to prove that pagination is unnecessary, so the response must be interpreted alongside the endpoint’s behaviour and the date coverage of the collected records.

In [15]:
# Test whether the endpoint responds differently to a page parameter
pagination_response = requests.get(
    api_url,
    params={"page": 2},
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

pagination_response.raise_for_status()

pagination_data = pagination_response.json()

print("Status code:", pagination_response.status_code)
print("Original record count:", len(api_data))
print("Response with page=2:", len(pagination_data))

if isinstance(pagination_data, list) and pagination_data:
    print("First record ID:", pagination_data[0].get("id"))
    print("Last record ID:", pagination_data[-1].get("id"))

Status code: 200
Original record count: 62333
Response with page=2: 62333
First record ID: 64143
Last record ID: 18847


## Comparing the Original and Paginated Responses

The request with the page=2 parameter returned the same number of records as the original request. I will compare the returned records to determine whether the parameter changes the dataset or whether the endpoint returns the same collection regardless of this parameter.

In [16]:
# Compare the original and page=2 responses
original_df = pd.DataFrame(api_data)
page_2_df = pd.DataFrame(pagination_data)

print("Same number of records:", len(original_df) == len(page_2_df))
print("Identical records:", original_df.equals(page_2_df))
print(
    "Same record IDs in the same order:",
    original_df["id"].equals(page_2_df["id"])
)

Same number of records: True
Identical records: True
Same record IDs in the same order: True


No Pagination behaviour was demonstrated by this test above

## Checking the Date Coverage of the Collected Data

The original request and the request with page=2 returned identical records. I will now inspect the date range and the number of unique dates represented in the collected data. This will help assess the historical coverage of the dataset before saving the raw collection.

In [17]:
# Inspect the date coverage of the collected records
ratedates = pd.to_datetime(df["ratedate"], errors="coerce")

print("Earliest date:", ratedates.min())
print("Latest date:", ratedates.max())
print("Unique dates:", ratedates.nunique())
print("Invalid dates:", ratedates.isna().sum())

Earliest date: 2001-12-10 00:00:00
Latest date: 2026-10-09 00:00:00
Unique dates: 6095
Invalid dates: 0


## Saving the Raw Exchange Rate Dataset

After collecting the exchange-rate records and inspecting the dataset structure and date coverage, I will save the original data to a CSV file. This preserves the collected records before cleaning and preprocessing, allowing the raw data to be reused and compared with the cleaned dataset.

In [18]:
# Save the original collected data
df.to_csv("cbn_exchange_rates_raw.csv", index=False)

print("Raw dataset saved successfully.")

Raw dataset saved successfully.


# Level 1 — Task 2: Data Cleaning and Preprocessing

The objective of this task is to clean and preprocess the collected exchange-rate dataset so that it is suitable for further analysis. I will inspect the data, identify quality issues, handle missing values, investigate invalid observations and outliers, encode categorical variables, and standardize numerical features where appropriate.

The raw dataset has been saved separately to preserve the original records before cleaning.

In [19]:
# Inspect the dataset structure and data types
df.info()

# Check for missing values in each column
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 62333 entries, 0 to 62332
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           62333 non-null  int64
 1   currency     62333 non-null  str  
 2   ratedate     62333 non-null  str  
 3   buyingrate   62333 non-null  str  
 4   centralrate  62333 non-null  str  
 5   sellingrate  62333 non-null  str  
dtypes: int64(1), str(5)
memory usage: 5.2 MB


id             0
currency       0
ratedate       0
buyingrate     0
centralrate    0
sellingrate    0
dtype: int64

No Missing Values where found from the above data check, this means no imputation is needed

## Inspecting Currency Labels

Before standardising the currency names, I will inspect the distinct labels in the original dataset. This will help me identify inconsistent capitalization, leading or trailing whitespace, and alternative names that may refer to the same currency.

I will examine the original labels first so that any subsequent corrections are based on the observed data rather than assumptions. No changes will be made to the currency column during this inspection.

In [20]:
# Inspect the original currency labels
currency_labels = df["currency"].unique()

print("Number of distinct labels:", df["currency"].nunique())
print("\nOriginal currency labels:")

for currency in sorted(currency_labels):
    print(repr(currency))

Number of distinct labels: 28

Original currency labels:
'CFA'
'CFA '
'DANISH KRONA'
'DANISH KRONER'
'EURO'
'EURO\t'
'EURO '
'JAPANESE YEN'
'NAIRA'
'POESO'
'POUND STERLING'
'POUNDS STERLING'
'RIYAL'
'RIYAL '
'SDR'
'SDR\t'
'SDR '
'SOUTH AFRICAN RAND'
'SWISS FRANC'
'SWISS FRANC\t'
'UAE DIRHAM'
'US DOLLAR'
'US DOLLAR '
'WAUA'
'WAUA '
'YEN'
'YEN '
'YUAN/RENMINBI'


## Standardising Currency Label Formatting

The inspection identified inconsistent whitespace in several currency labels. I will remove leading and trailing whitespace, including tabs, and convert the labels to uppercase. These changes address formatting inconsistencies without merging different currency names.

After standardising the formatting, I will inspect the distinct labels again before deciding whether any alternative names should be mapped to a common label.

In [21]:
# Standardise whitespace and capitalization
df["currency"] = df["currency"].str.strip().str.upper()

# Inspect the labels after formatting
print("Distinct currency labels:", df["currency"].nunique())
display(df["currency"].value_counts().sort_index())

Distinct currency labels: 18


currency
CFA                   6048
DANISH KRONA          3463
DANISH KRONER         1092
EURO                  6071
JAPANESE YEN             1
NAIRA                    8
POESO                    3
POUND STERLING           5
POUNDS STERLING       6075
RIYAL                 5535
SDR                   4536
SOUTH AFRICAN RAND    2353
SWISS FRANC           5118
UAE DIRHAM             170
US DOLLAR             6082
WAUA                  6072
YEN                   6072
YUAN/RENMINBI         3629
Name: count, dtype: int64

## Investigating Unusual Currency Labels

After standardising whitespace and capitalization, several currency labels remain that may be alternative names or unusual entries. I will inspect their associated records before deciding whether any labels should be corrected or merged. This avoids making unsupported changes to the original data.

In [22]:
# Inspect records associated with unusual currency labels
unusual_labels = [
    "DANISH KRONER",
    "JAPANESE YEN",
    "POUND STERLING",
    "NAIRA",
    "WAUA",
    "POESO"
]

display(
    df.loc[
        df["currency"].isin(unusual_labels),
        ["id", "currency", "ratedate", "buyingrate", "centralrate", "sellingrate"]
    ].sort_values(["currency", "ratedate"])
)

,id,currency,ratedate,buyingrate,centralrate,sellingrate
51673,2578,DANISH KRONER,2008-03-07,23.9319,24.0352,24.1386
51666,2579,DANISH KRONER,2008-03-10,23.9319,24.0352,24.1386
51653,2580,DANISH KRONER,2008-03-11,23.8495,23.9525,24.0555
51646,2581,DANISH KRONER,2008-03-12,23.8434,23.9464,24.0494
51633,2582,DANISH KRONER,2008-03-13,24.1528,24.2572,24.3615
...,...,...,...,...,...,...
63,64101,WAUA,2026-10-05,1799.0281,1799.7041,1800.3800
50,64114,WAUA,2026-10-06,1799.5172,1800.1938,1800.8704
37,64128,WAUA,2026-10-07,1800.7455,1801.4221,1802.0987
24,64141,WAUA,2026-10-08,1798.7758,1799.4515,1800.1271


## Correcting Verified Currency Name Variations

The CBN’s published historical exchange-rate data uses the labels DANISH KRONA, POUNDS STERLING, and YEN. I will map the observed alternative labels to these source-supported names to ensure consistent currency categories. Other unusual labels will remain unchanged until their meanings can be verified.

In [23]:
# Standardise verified currency-name variations
df["currency"] = df["currency"].replace({
    "DANISH KRONER": "DANISH KRONA",
    "JAPANESE YEN": "YEN",
    "POUND STERLING": "POUNDS STERLING"
})

# Check the updated currency labels
print("Distinct currency labels:", df["currency"].nunique())
display(df["currency"].value_counts().sort_index())

Distinct currency labels: 15


currency
CFA                   6048
DANISH KRONA          4555
EURO                  6071
NAIRA                    8
POESO                    3
POUNDS STERLING       6080
RIYAL                 5535
SDR                   4536
SOUTH AFRICAN RAND    2353
SWISS FRANC           5118
UAE DIRHAM             170
US DOLLAR             6082
WAUA                  6072
YEN                   6073
YUAN/RENMINBI         3629
Name: count, dtype: int64

## Creating a Separate Dataset for Cleaning

I will create a copy of the collected dataset for cleaning and preprocessing. This keeps the cleaning operations separate from the original dataset saved as a raw CSV file.

In [24]:
df_clean = df.copy()

print("Original dataset shape:", df.shape)
print("Cleaning dataset shape:", df_clean.shape)

Original dataset shape: (62333, 6)
Cleaning dataset shape: (62333, 6)


## Checking for Duplicate Exchange-Rate Records

I will check for duplicate records using the currency, rate date, buying rate, central rate, and selling rate columns, excluding the unique record ID. This helps identify records that contain identical exchange-rate information even when their IDs differ.

I will inspect the number of duplicates before removing any records to avoid deleting valid historical observations.

In [25]:
duplicate_columns = [
    "currency",
    "ratedate",
    "buyingrate",
    "centralrate",
    "sellingrate"
]

print(
    "Duplicate exchange-rate records:",
    df_clean.duplicated(subset=duplicate_columns).sum()
)

Duplicate exchange-rate records: 37


## Inspecting Duplicate Exchange-Rate Records

I will display the records identified as duplicates to confirm that their exchange-rate information matches. This inspection will help determine whether the records can be safely removed without losing distinct historical information.

In [26]:
duplicate_records = df_clean[
    df_clean.duplicated(subset=duplicate_columns, keep=False)
].sort_values(duplicate_columns)

display(duplicate_records)

,id,currency,ratedate,buyingrate,centralrate,sellingrate
37518,24926,CFA,2013-10-11,0.3003,0.3103,0.3203
37519,24927,CFA,2013-10-11,0.3003,0.3103,0.3203
10617,52285,CFA,2023-03-17,0.7237,0.7337,0.7437
10618,52286,CFA,2023-03-17,0.7237,0.7337,0.7437
6305,56718,CFA,2024-09-11,2.6908,2.7008,2.7108
...,...,...,...,...,...,...
12495,50340,YUAN/RENMINBI,2022-07-29,61.5198,61.6686,61.8174
8738,54233,YUAN/RENMINBI,2023-11-14,115.8237,115.8924,115.9611
8739,54234,YUAN/RENMINBI,2023-11-14,115.8237,115.8924,115.9611
6306,56719,YUAN/RENMINBI,2024-09-11,222.9554,223.0256,223.0958


## Removing Exact Duplicate Records

After inspecting the duplicate records, I will remove redundant rows that have identical currency, date, buying rate, central rate, and selling rate values. One copy of each unique exchange-rate record will be retained.

In [27]:
rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates(
    subset=duplicate_columns,
    keep="first"
).copy()

rows_removed = rows_before - len(df_clean)

print("Rows before removing duplicates:", rows_before)
print("Duplicate rows removed:", rows_removed)
print("Rows after removing duplicates:", len(df_clean))

Rows before removing duplicates: 62333
Duplicate rows removed: 37
Rows after removing duplicates: 62296


In [28]:
# Investigate the less common currency labels 

unusual_currencies = ["NAIRA", "POESO", "WAUA"]

for currency in unusual_currencies:
    print(f"\nRecords for {currency}:")
    display(
        df_clean.loc[
            df_clean["currency"].eq(currency),
            ["id", "currency", "ratedate",
             "buyingrate", "centralrate", "sellingrate"]
        ].sort_values("ratedate")
    )


Records for NAIRA:


,id,currency,ratedate,buyingrate,centralrate,sellingrate
50061,6249,NAIRA,2008-10-31,100.1473,100.5806,101.0139
49722,6250,NAIRA,2008-12-23,0.2585,0.2685,0.2785
46189,6251,NAIRA,2010-06-04,1.5961,1.6015,1.6069
46093,6252,NAIRA,2010-06-17,182.7173,183.3358,183.9543
46089,6253,NAIRA,2010-06-18,216.7064,217.4400,218.1735
45974,6254,NAIRA,2010-07-05,24.7807,24.8647,24.9487
45948,6255,NAIRA,2010-07-08,0.2637,0.2737,0.2837
41239,6256,NAIRA,2012-05-31,154.7500,155.2500,155.7500



Records for POESO:


,id,currency,ratedate,buyingrate,centralrate,sellingrate
57697,6257,POESO,2004-12-10,251.2006,252.1532,253.1058
57361,6258,POESO,2005-02-21,249.7428,250.6898,251.6368
48452,6259,POESO,2009-07-01,27.5254,27.6199,27.7143



Records for WAUA:


,id,currency,ratedate,buyingrate,centralrate,sellingrate
62330,16249,WAUA,2001-12-10,138.8876,139.5084,140.1293
62323,16250,WAUA,2001-12-11,138.8876,139.5084,140.1293
62318,16251,WAUA,2001-12-12,138.8876,139.5084,140.1293
62311,16252,WAUA,2001-12-13,138.8876,139.5084,140.1293
62306,16253,WAUA,2001-12-14,139.3264,139.7279,140.1293
...,...,...,...,...,...,...
63,64101,WAUA,2026-10-05,1799.0281,1799.7041,1800.3800
50,64114,WAUA,2026-10-06,1799.5172,1800.1938,1800.8704
37,64128,WAUA,2026-10-07,1800.7455,1801.4221,1802.0987
24,64141,WAUA,2026-10-08,1798.7758,1799.4515,1800.1271


## Review of Unusual Currency Labels

An investigation identified three less common currency labels: NAIRA (8 records), POESO (3 records), and WAUA (6,065 records). The records were inspected to review their dates and exchange-rate values.

The available dataset does not independently establish whether the labels NAIRA and POESO are valid source categories or data-entry anomalies. The WAUA label occurs throughout the historical dataset, but its meaning and rate values have not been independently verified.

Decision: These records are retained without modification because there is insufficient evidence to justify deleting, renaming, or correcting them. The uncertainty is documented as a limitation of the collected data.

## Checking for Conflicting Rates on the Same Date

After removing exact duplicate records, I will check whether any currency has multiple records for the same date with different exchange rates. These records require investigation before deciding whether any should be removed, so that valid historical information is not lost.

In [29]:
date_currency_duplicates = df_clean[
    df_clean.duplicated(
        subset=["currency", "ratedate"],
        keep=False
    )
].sort_values(["currency", "ratedate", "id"])

display(date_currency_duplicates)

,id,currency,ratedate,buyingrate,centralrate,sellingrate
7941,55057,CFA,2024-02-22,2.6244,2.6344,2.6444
7953,55069,CFA,2024-02-22,2.6261,2.6361,2.6461
6924,56094,CFA,2024-07-02,2.4521,2.4621,2.4721
6936,56106,CFA,2024-07-02,2.4485,2.4585,2.4685
6594,56430,CFA,2024-08-08,2.6318,2.6418,2.6518
...,...,...,...,...,...,...
6895,56136,YUAN/RENMINBI,2024-07-04,207.3069,207.3757,207.4444
3182,59918,YUAN/RENMINBI,2025-10-06,206.3540,206.4243,206.4945
3187,59923,YUAN/RENMINBI,2025-10-06,85.1593,85.1883,85.2172
3074,60027,YUAN/RENMINBI,2025-10-17,206.9265,206.9967,207.0669


## Decision on Repeated Currency-Date Records

The inspection identified multiple records for the same currency and date with different buying, central, or selling rates. Repeated dates are not automatically errors, because multiple observations can be valid depending on the structure of the source dataset.

At this stage, I will retain these records without modification. The CBN source information examined so far does not establish which record should be preferred when rates differ for the same currency and date. Therefore, I will not arbitrarily select the earliest record, the latest record, or the record with the highest ID.

The unusually large differences between some of these records will be investigated separately as part of the exchange-rate validation process. Any decision to correct or remove a record will be based on supporting evidence rather than the repeated date alone.

## Converting Dates and Exchange Rates to Appropriate Data Types

I will convert the rate date column to datetime and the buying, central, and selling rate columns to numeric values. This will allow accurate date sorting, numerical comparisons, and identification of invalid values during subsequent cleaning.

In [30]:
# Convert the date column to datetime
df_clean["ratedate"] = pd.to_datetime(
    df_clean["ratedate"],
    errors="coerce"
)

# Convert exchange-rate columns to numeric
rate_columns = ["buyingrate", "centralrate", "sellingrate"]

for column in rate_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

# Summarise the resulting data types and invalid values
print(df_clean.dtypes)
print("\nMissing or invalid values after conversion:")
display(df_clean[["ratedate"] + rate_columns].isna().sum())

id                      int64
currency                  str
ratedate       datetime64[us]
buyingrate            float64
centralrate           float64
sellingrate           float64
dtype: object

Missing or invalid values after conversion:


ratedate       0
buyingrate     0
centralrate    0
sellingrate    0
dtype: int64

## Investigating Unusual Exchange-Rate Values

I will examine records where the buying, central, and selling rates differ substantially in scale or where the rates are inconsistent with one another. This will help identify suspicious observations for further investigation before any corrections or removals are made.

In [31]:
# Check whether the three rates follow the expected order
rate_order_violations = df_clean[
    (df_clean["buyingrate"] > df_clean["centralrate"])
    | (df_clean["centralrate"] > df_clean["sellingrate"])
]

print("Records violating the expected rate order:", len(rate_order_violations))

display(
    rate_order_violations[
        ["id", "currency", "ratedate", "buyingrate", "centralrate", "sellingrate"]
    ].head(20)
)

Records violating the expected rate order: 76


,id,currency,ratedate,buyingrate,centralrate,sellingrate
3412,59694,SOUTH AFRICAN RAND,2025-09-08,85.8887,85.1972,95.9457
3781,59317,DANISH KRONA,2025-07-24,241.8920,241.4539,242.0497
3995,59094,YUAN/RENMINBI,2025-06-27,214.6951,214.5327,214.6054
4163,58930,EURO,2025-06-04,1787.9984,1787.9697,1788.5409
4426,58668,YEN,2025-05-05,11.1619,11.1563,11.1688
9451,53500,YEN,2023-08-18,5.1190,5.1124,5.1259
10387,52513,POUNDS STERLING,2023-04-17,572.4170,572.0382,572.6594
10428,52481,WAUA,2023-04-12,619.1547,617.8278,620.5009
11095,51795,SOUTH AFRICAN RAND,2023-01-20,66.1713,66.2440,26.4253
11295,51569,CFA,2022-12-23,0.7040,0.0714,0.7240


## Findings and Decision on Rate-Order Violations

The rate-order check identified 76 records in which the buying rate exceeds the central rate or the central rate exceeds the selling rate.

This check is a screening method, not proof that every flagged record is erroneous. Some records may contain genuine source-specific differences, while others may reflect data-quality issues.

At this stage, I will retain all flagged records without modification. I will investigate unusually large differences between the rates and validate suspicious observations before deciding whether any values should be corrected or removed.

No records have been removed or altered as a result of this check.

## Assessing the Severity of Rate-Order Violations

I will calculate the absolute differences between the buying and central rates, and between the selling and central rates, for the records that violate the expected rate order. This will help prioritise observations for investigation without automatically assuming that every violation is an error.

In [32]:
# Calculate rate differences for the flagged records
rate_order_violations = rate_order_violations.copy()

rate_order_violations["buying_central_difference"] = (
    rate_order_violations["buyingrate"]
    - rate_order_violations["centralrate"]
).abs()

rate_order_violations["selling_central_difference"] = (
    rate_order_violations["sellingrate"]
    - rate_order_violations["centralrate"]
).abs()

display(
    rate_order_violations[
        [
            "id",
            "currency",
            "ratedate",
            "buyingrate",
            "centralrate",
            "sellingrate",
            "buying_central_difference",
            "selling_central_difference"
        ]
    ].sort_values(
        "buying_central_difference",
        ascending=False
    ).head(15)
)

,id,currency,ratedate,buyingrate,centralrate,sellingrate,buying_central_difference,selling_central_difference
29244,33232,YEN,2016-11-01,29009.0000,2.9057,2.9105,29006.0943,0.0048
30222,32251,YEN,2016-06-21,27076.0000,2.7124,2.7172,27073.2876,0.0048
40335,22073,YEN,2012-09-27,19920.0000,1.9985,2.0049,19918.0015,0.0064
13842,48970,SDR,2022-02-11,5843.8960,584.5980,585.2999,5259.2980,0.7019
33522,28948,CFA,2015-04-02,3022.0000,0.3122,0.3222,3021.6878,0.0100
61848,81,CFA,2002-04-10,0.1549,1556.0000,0.1563,1555.8451,1555.8437
59395,14138,US DOLLAR,2003-12-09,136.3000,1376.8000,137.3000,1240.5000,1239.5000
61951,16312,WAUA,2002-03-14,1240.1985,140.8076,141.4166,1099.3909,0.6090
35089,27388,SDR,2014-09-04,234.1374,23.8940,235.6506,210.2434,211.7566
61729,3754,EURO,2002-05-09,140.5079,104.9611,105.4143,35.5468,0.4532


## Flagging Extreme Differences Between Exchange Rates

I will flag records where the largest of the buying, central, and selling rates is more than five times the smallest. Since these three rates for the same currency and date would ordinarily be expected to be relatively close, large differences may indicate data-entry or source-quality problems. Flagged records will be investigated before any values are changed or removed.

In [33]:
rate_columns = ["buyingrate", "centralrate", "sellingrate"]

df_clean["rate_max_min_ratio"] = (
    df_clean[rate_columns].max(axis=1)
    / df_clean[rate_columns].min(axis=1).where(
        df_clean[rate_columns].min(axis=1) > 0
    )
)

extreme_rate_records = df_clean[
    df_clean["rate_max_min_ratio"] > 5
].sort_values("rate_max_min_ratio", ascending=False)

print("Records flagged for investigation:", len(extreme_rate_records))

display(
    extreme_rate_records[
        [
            "id", "currency", "ratedate",
            "buyingrate", "centralrate", "sellingrate",
            "rate_max_min_ratio"
        ]
    ].head(20).round(4)
)

Records flagged for investigation: 43


C:\Users\user\AppData\Local\Temp\ipykernel_15836\416518889.py:23: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].head(20).round(4)


,id,currency,ratedate,buyingrate,centralrate,sellingrate,rate_max_min_ratio
46195,2059,CFA,2010-06-03,0.2526,1.381126e+03,2.762000e+03,10934.2835
61797,90,CFA,2002-04-23,0.1549,1.556000e-01,1.563000e+03,10090.3809
61969,3714,EURO,2002-03-11,100.6780,1.011153e+02,1.015527e+06,10086.8809
55957,4636,EURO,2005-12-05,149.8624,7.552409e+05,1.510332e+06,10078.1250
49719,2773,DANISH KRONA,2008-12-23,24.4385,1.231432e+05,2.462620e+05,10076.8050
57003,17087,WAUA,2005-05-10,198.0008,9.976145e+05,1.995031e+06,10075.8734
58004,6955,POUNDS STERLING,2004-10-08,235.6300,1.187201e+06,2.374167e+06,10075.8265
49086,18029,WAUA,2009-03-30,215.3712,1.084403e+06,2.168591e+06,10069.0854
40323,22083,SWISS FRANC,2012-09-28,165.3986,1.659329e+02,1.664672e+06,10064.6076
61848,81,CFA,2002-04-10,0.1549,1.556000e+03,1.563000e-01,10045.1904


## Findings and Decision on Extreme Rate Differences

The screening check flagged 43 records in which the largest of the buying, central, and selling rates was more than five times the smallest rate.

The results include substantial discrepancies within individual records, such as unusually large buying rates or central and selling rates that differ greatly from the other rate values. These observations may indicate data-entry, formatting, or source-quality issues.

The five-times threshold is an investigative rule used for this project, not an official CBN validation rule. I will retain the flagged records until the suspicious values can be validated. No rates have been corrected or removed at this stage.

## Validating Suspicious Exchange Rates Against Nearby Observations

The screening process identified 43 records with unusually large differences between their buying, central, and selling rates. I will compare the most extreme observations with nearby dates for the same currency to determine whether the discrepancy appears isolated or persists over time.

This comparison will help prioritise records for further investigation. It will not, by itself, prove that a value is incorrect or justify automatically changing it.

In [34]:
# Sort records chronologically within each currency
df_sorted = df_clean.sort_values(
    ["currency", "ratedate"]
).copy()

# Add the previous and next central rates for each currency
df_sorted["previous_date"] = (
    df_sorted.groupby("currency")["ratedate"].shift(1)
)

df_sorted["previous_centralrate"] = (
    df_sorted.groupby("currency")["centralrate"].shift(1)
)

df_sorted["next_date"] = (
    df_sorted.groupby("currency")["ratedate"].shift(-1)
)

df_sorted["next_centralrate"] = (
    df_sorted.groupby("currency")["centralrate"].shift(-1)
)

# Select the 10 most extreme records for investigation
extreme_records = (
    df_sorted[df_sorted["rate_max_min_ratio"] > 5]
    .nlargest(10, "rate_max_min_ratio")
)

display(
    extreme_records[
        [
            "id", "currency", "previous_date", "previous_centralrate",
            "ratedate", "buyingrate", "centralrate", "sellingrate",
            "next_date", "next_centralrate", "rate_max_min_ratio"
        ]
    ].reset_index(drop=True)
)

,id,currency,previous_date,previous_centralrate,ratedate,buyingrate,centralrate,sellingrate,next_date,next_centralrate,rate_max_min_ratio
0,2059,CFA,2010-06-02,0.2648,2010-06-03,0.2526,1.381126e+03,2.762000e+03,2010-06-04,0.2662,10934.283452
1,90,CFA,2002-04-22,0.1556,2002-04-23,0.1549,1.556000e-01,1.563000e+03,2002-04-24,0.1556,10090.380891
2,3714,EURO,2002-03-08,101.6240,2002-03-11,100.6780,1.011153e+02,1.015527e+06,2002-03-12,100.8841,10086.880947
3,4636,EURO,2005-12-02,150.4724,2005-12-05,149.8624,7.552409e+05,1.510332e+06,2005-12-06,151.3486,10078.125000
4,2773,DANISH KRONA,2008-12-22,24.5131,2008-12-23,24.4385,1.231432e+05,2.462620e+05,2008-12-24,24.5859,10076.805041
5,17087,WAUA,2005-05-09,199.8630,2005-05-10,198.0008,9.976145e+05,1.995031e+06,2005-05-11,198.8377,10075.873431
6,6955,POUNDS STERLING,2004-10-07,235.6364,2004-10-08,235.6300,1.187201e+06,2.374167e+06,2004-10-11,237.7015,10075.826508
7,18029,WAUA,2009-03-27,218.4349,2009-03-30,215.3712,1.084403e+06,2.168591e+06,2009-03-31,215.6914,10069.085374
8,22083,SWISS FRANC,2012-09-27,165.2091,2012-09-28,165.3986,1.659329e+02,1.664672e+06,2012-10-02,165.8974,10064.607560
9,81,CFA,2002-04-09,0.1556,2002-04-10,0.1549,1.556000e+03,1.563000e-01,2002-04-11,0.1556,10045.190445


## Findings from Comparing Nearby Exchange-Rate Observations

The comparison of suspicious records with adjacent observations revealed several isolated, unusually large rate values. For example, the CFA central rate on 3 June 2010 is approximately 1,381.126, compared with 0.2648 on the preceding date and 0.2662 on the following date.

Similar abrupt differences appear in other currencies. These patterns strengthen the case for investigating the flagged observations, but neighbouring values alone do not establish the correct replacement values.

I will not automatically interpolate, rescale, or delete these records. The next step is to inspect the corresponding records in the saved raw dataset to establish whether the anomalies were already present in the collected data or arose during preprocessing.

In [35]:
df_raw_check = pd.read_csv("cbn_exchange_rates_raw.csv")

suspect_ids = [2059, 90, 3714, 4636, 2773, 17087, 6955, 18029, 22083, 81]

display(
    df_raw_check.loc[
        df_raw_check["id"].isin(suspect_ids),
        ["id", "currency", "ratedate", "buyingrate", "centralrate", "sellingrate"]
    ].sort_values("id")
)

,id,currency,ratedate,buyingrate,centralrate,sellingrate
61848,81,CFA,2002-04-10,0.1549,1.556000e+03,1.563000e-01
61797,90,CFA,2002-04-23,0.1549,1.556000e-01,1.563000e+03
46195,2059,CFA,2010-06-03,0.2526,1.381126e+03,2.762000e+03
49719,2773,DANISH KRONER,2008-12-23,24.4385,1.231432e+05,2.462620e+05
61969,3714,EURO,2002-03-11,100.6780,1.011153e+02,1.015527e+06
55957,4636,EURO,2005-12-05,149.8624,7.552409e+05,1.510332e+06
58004,6955,POUNDS STERLING,2004-10-08,235.6300,1.187201e+06,2.374167e+06
57003,17087,WAUA,2005-05-10,198.0008,9.976145e+05,1.995031e+06
49086,18029,WAUA,2009-03-30,215.3712,1.084403e+06,2.168591e+06
40323,22083,SWISS FRANC,2012-09-28,165.3986,1.659329e+02,1.664672e+06


## Findings from Inspecting the Saved Raw Data

The suspicious exchange-rate values were also present in the saved raw CSV file, which was created before the cleaning and numerical conversion steps. This indicates that the observed anomalies were not introduced by those later preprocessing operations.

Several records contain unusually large values in one or more rate columns compared with observations for the same currency on adjacent dates. For example, the CFA central rate recorded on 3 June 2010 is approximately 1,381.126, compared with 0.2648 on the preceding date and 0.2662 on the following date.

These observations are strong candidates for further investigation, but the saved raw data alone does not establish the correct values or independently verify the original CBN records.

Decision: Retain the original values for now, flag the suspicious observations for review, and avoid arbitrary corrections, deletion, or interpolation until sufficient evidence is available.

## Checking for Zero and Negative Exchange Rates

Exchange rates in this dataset represent the value of one currency relative to the Nigerian naira. Under the normal interpretation of these quoted rates, the buying rate, central rate, and selling rate should all be positive.

A zero or negative value may indicate a data-quality issue, such as an incorrect entry or an anomaly in the source data. However, a flagged value should be investigated rather than automatically deleted or replaced.

This check examines all three exchange-rate columns to identify records containing at least one zero or negative value. The results will help determine whether any observations require further investigation before the remaining cleaning and preprocessing steps.

In [36]:
rate_columns = ["buyingrate", "centralrate", "sellingrate"]

non_positive_rates = df_clean[
    (df_clean[rate_columns] <= 0).any(axis=1)
]

print("Records with at least one zero or negative rate:",
      len(non_positive_rates))

display(
    non_positive_rates[
        ["id", "currency", "ratedate",
         "buyingrate", "centralrate", "sellingrate"]
    ].sort_values(["currency", "ratedate"])
)

Records with at least one zero or negative rate: 1


,id,currency,ratedate,buyingrate,centralrate,sellingrate
54775,1131,CFA,2006-07-20,-0.231,0.005,0.241


## Findings from the Zero and Negative Rate Check

The check identified one record containing a non-positive exchange rate. The CFA record dated 20 July 2006 (ID 1131) has a buying rate of -0.231, a central rate of 0.005, and a selling rate of 0.241.

The negative buying rate is inconsistent with the normal interpretation of the exchange rates in this dataset and indicates a potential data-quality issue. However, the correct value cannot be established from this check alone.

Decision: The original values will be retained in df_clean temporarily while the record is investigated against the source data. Once the issue has been verified, the appropriate action will be determined: correction, treatment as a missing value, or exclusion from the final analysis. Any change will be documented and supported by evidence.

In [37]:
df_raw_check.loc[
    df_raw_check["id"] == 1131,
    ["id", "currency", "ratedate",
     "buyingrate", "centralrate", "sellingrate"]
]

,id,currency,ratedate,buyingrate,centralrate,sellingrate
54775,1131,CFA,2006-07-20,-0.231,0.005,0.241


## Investigating the Negative Buying Rate

The inspection of the saved raw dataset confirmed that the CFA record dated 20 July 2006 (ID 1131) contains a buying rate of -0.231. The same value was present before the numerical conversion and other preprocessing operations.

The record requires further investigation to determine whether the negative value is an isolated anomaly or part of a broader inconsistency in the historical series. The surrounding observations will be examined to provide additional context.

Decision: The record remains flagged for review. No correction or deletion will be made based solely on the negative value or the neighbouring observations.

In [38]:
cfa_nearby = (
    df_clean[
        (df_clean["currency"] == "CFA")
        & (df_clean["ratedate"].between("2006-07-17", "2006-07-24"))
    ]
    .sort_values("ratedate")
)

display(
    cfa_nearby[
        ["id", "currency", "ratedate",
         "buyingrate", "centralrate", "sellingrate"]
    ]
)

,id,currency,ratedate,buyingrate,centralrate,sellingrate
54802,1128,CFA,2006-07-17,0.2346,0.2396,0.2446
54791,1129,CFA,2006-07-18,0.2346,0.2396,0.2446
54786,1130,CFA,2006-07-19,0.2319,0.2369,0.2419
54775,1131,CFA,2006-07-20,-0.2310,0.0050,0.2410
54770,1132,CFA,2006-07-21,0.2341,0.2391,0.2441
54759,1133,CFA,2006-07-24,0.2341,0.2391,0.2441


## Findings from the Investigation of the Negative Buying Rate

The comparison of neighbouring CFA observations revealed that the record dated 20 July 2006 contains unusually low and negative values relative to the surrounding dates. The buying rate is -0.2310 and the central rate is 0.0050, while the selling rate is 0.2410.

The neighbouring observations show buying rates around 0.2319–0.2346 and central rates around 0.2369–0.2396. This supports the conclusion that the record contains a likely data-quality anomaly. However, the exact intended values cannot be established from neighbouring observations alone.

Decision: The record will remain flagged as an unresolved anomaly. The original values will be preserved until the intended values can be verified from a reliable source. No replacement values will be inferred from the surrounding dates.

## Consolidating Suspicious Exchange-Rate Records

Previous checks identified records with unusual rate ordering, extreme differences between exchange-rate columns, and a negative buying rate. These checks serve different purposes, and some flagged records may overlap.

The flagged observations will be consolidated for review to provide a clearer picture of the potential data-quality issues. This will help distinguish records requiring further validation from those that can be retained without changes.

Decision: The review table will be used for investigation and documentation only. The original exchange-rate values in df_clean will remain unchanged until any proposed correction or exclusion is supported by evidence.

In [39]:
rate_columns = ["buyingrate", "centralrate", "sellingrate"]

review_records = df_clean[
    (df_clean[rate_columns] <= 0).any(axis=1)
    | (
        df_clean[rate_columns].max(axis=1)
        / df_clean[rate_columns].min(axis=1).where(
            df_clean[rate_columns].min(axis=1) > 0
        )
    ).gt(5)
    | (
        (df_clean["buyingrate"] > df_clean["centralrate"])
        | (df_clean["centralrate"] > df_clean["sellingrate"])
    )
].copy()

review_records = review_records.sort_values(
    ["currency", "ratedate", "id"]
)

print("Unique records flagged for review:", len(review_records))

display(
    review_records[
        ["id", "currency", "ratedate",
         "buyingrate", "centralrate", "sellingrate"]
    ]
)

Unique records flagged for review: 105


,id,currency,ratedate,buyingrate,centralrate,sellingrate
61848,81,CFA,2002-04-10,0.1549,1556.0000,0.1563
61797,90,CFA,2002-04-23,0.1549,0.1556,1563.0000
61437,150,CFA,2002-07-19,0.1872,0.1880,0.1187
61272,177,CFA,2002-08-27,0.1944,0.1952,196.0000
60780,259,CFA,2002-12-31,0.1830,0.1837,0.1644
...,...,...,...,...,...,...
40741,21617,YUAN/RENMINBI,2012-08-03,24.9930,24.3782,24.4571
28961,33521,YUAN/RENMINBI,2016-12-07,44.3724,44.3013,44.3744
20709,41829,YUAN/RENMINBI,2019-09-27,42.9148,42.9853,42.0559
14216,48572,YUAN/RENMINBI,2021-12-24,64.5327,64.6117,54.6906


## Excluding the Invalid CFA Exchange-Rate Record

Record ID 1131, dated 20 July 2006, would now be excluded from the cleaned dataset because its buying rate is negative and its central rate is inconsistent with surrounding CFA observations. The exact intended values cannot be established confidently from the available data, so the record is excluded rather than corrected using estimated values. The original record remains preserved in the raw CSV file for traceability.

In [40]:
rows_before = len(df_clean)

df_clean = df_clean[df_clean["id"] != 1131].copy()

rows_removed = rows_before - len(df_clean)

print("Rows removed:", rows_removed)
print("Rows remaining:", len(df_clean))

Rows removed: 1
Rows remaining: 62295


## Summary of Exchange-Rate Quality Flags

The suspicious exchange-rate records are summarized by screening rule and currency to clarify the types of potential data-quality issues present in the dataset. The checks cover non-positive rates, unusually large differences among the three quoted rates, and violations of the expected buying–central–selling rate order.

These rules identify records requiring attention; they do not automatically establish that every flagged observation is incorrect. The summary supports decisions about further investigation while avoiding the unnecessary removal of potentially valid data.

In [41]:
rate_columns = ["buyingrate", "centralrate", "sellingrate"]

rate_min = df_clean[rate_columns].min(axis=1)
rate_max = df_clean[rate_columns].max(axis=1)

flag_summary = pd.DataFrame({
    "non_positive_rate": (df_clean[rate_columns] <= 0).any(axis=1),
    "extreme_rate_difference": (
        rate_max / rate_min.where(rate_min > 0)
    ).gt(5),
    "rate_order_violation": (
        (df_clean["buyingrate"] > df_clean["centralrate"])
        | (df_clean["centralrate"] > df_clean["sellingrate"])
    )
}, index=df_clean.index)

print("Records flagged by screening reason:")
display(flag_summary.sum().rename("record_count").to_frame())

print("Flagged records by currency and screening reason:")
currency_summary = (
    pd.concat([df_clean[["currency"]], flag_summary], axis=1)
    .groupby("currency")
    .sum()
    .astype(int)
)

display(currency_summary)

print("Unique records flagged:", flag_summary.any(axis=1).sum())

Records flagged by screening reason:


,record_count
non_positive_rate,0
extreme_rate_difference,43
rate_order_violation,76


Flagged records by currency and screening reason:


,non_positive_rate,extreme_rate_difference,rate_order_violation
currency,,,
CFA,0,14,12
DANISH KRONA,0,2,2
EURO,0,4,7
NAIRA,0,0,0
POESO,0,0,0
POUNDS STERLING,0,1,8
RIYAL,0,2,2
SDR,0,4,4
SOUTH AFRICAN RAND,0,0,7


Unique records flagged: 104


## Removing Extreme Exchange-Rate Outliers

The screening identifies records in which the largest quoted rate is more than five times the smallest positive quoted rate. Such large differences can indicate data-quality problems, particularly when the values also differ sharply from observations on neighbouring dates.

The ratio threshold is used as a project-specific screening rule, not an official CBN standard. Records meeting this criterion are excluded from the analysis-ready dataset because the extreme differences make them unsuitable for reliable analysis without further verification. The original raw CSV remains unchanged, and the exclusion rule is documented for reproducibility.

Limitation: This rule may not identify every anomalous observation, and the ratio alone does not establish the correct exchange-rate values.

In [42]:
rate_columns = ["buyingrate", "centralrate", "sellingrate"]

rate_min = df_clean[rate_columns].min(axis=1)
rate_max = df_clean[rate_columns].max(axis=1)

extreme_rate_difference = (
    rate_max / rate_min.where(rate_min > 0)
).gt(5)

rows_before = len(df_clean)

df_clean = df_clean.loc[~extreme_rate_difference].copy()

rows_removed = rows_before - len(df_clean)

print("Extreme-rate records removed:", rows_removed)
print("Rows remaining:", len(df_clean))

Extreme-rate records removed: 43
Rows remaining: 62252


## Findings from the Exchange-Rate Quality Checks

The initial screening identified 104 unique records requiring review. Following the documented exclusion of the invalid CFA record and the removal of 43 records meeting the extreme rate-difference rule, the cleaned dataset contains 62,252 records.

The remaining dataset will be reassessed using the non-positive-rate, extreme rate-difference, and rate-order screening rules. These checks help identify potential data-quality issues that may require further investigation.

Decision: The 43 extreme-rate records are excluded under the documented project-specific screening rule. The remaining flagged records are retained pending further review because screening flags alone do not establish that an observation is incorrect. The original raw CSV remains unchanged.

In [43]:
rate_columns = ["buyingrate", "centralrate", "sellingrate"]

rate_min = df_clean[rate_columns].min(axis=1)
rate_max = df_clean[rate_columns].max(axis=1)

flag_summary = pd.DataFrame({
    "non_positive_rate": (df_clean[rate_columns] <= 0).any(axis=1),
    "extreme_rate_difference": (
        rate_max / rate_min.where(rate_min > 0)
    ).gt(5),
    "rate_order_violation": (
        (df_clean["buyingrate"] > df_clean["centralrate"])
        | (df_clean["centralrate"] > df_clean["sellingrate"])
    )
}, index=df_clean.index)

print("Records flagged by screening reason:")
display(flag_summary.sum().rename("record_count").to_frame())

print("Flagged records by currency and screening reason:")
currency_summary = (
    pd.concat([df_clean[["currency"]], flag_summary], axis=1)
    .groupby("currency")
    .sum()
    .astype(int)
)

display(currency_summary)

print("Unique records flagged:", flag_summary.any(axis=1).sum())

Records flagged by screening reason:


,record_count
non_positive_rate,0
extreme_rate_difference,0
rate_order_violation,61


Flagged records by currency and screening reason:


,non_positive_rate,extreme_rate_difference,rate_order_violation
currency,,,
CFA,0,0,9
DANISH KRONA,0,0,2
EURO,0,0,6
NAIRA,0,0,0
POESO,0,0,0
POUNDS STERLING,0,0,8
RIYAL,0,0,2
SDR,0,0,1
SOUTH AFRICAN RAND,0,0,7


Unique records flagged: 61


## Final Data-Quality Review and Decisions

Following the exclusion of the invalid CFA record and the removal of 43 records meeting the extreme rate-difference rule, the cleaned dataset contains 62,252 records.

The final screening identifies 61 records that violate the expected buying–central–selling rate order. No records trigger the non-positive-rate or extreme rate-difference rules in the current dataset.

Final decision: The 43 extreme-rate records and the invalid CFA record are excluded from the cleaned dataset. The 61 remaining rate-order violations are retained because a violation alone does not establish that an observation is incorrect. These records remain documented as potential data-quality issues. The original raw CSV remains unchanged.

## Removing the Temporary Outlier-Diagnostic Column

The rate_max_min_ratio column was created temporarily to identify records with unusually large differences among their buying, central, and selling rates. It served its purpose during the outlier investigation and is not part of the original dataset.

Decision: The temporary diagnostic column is removed from df_clean to retain only the relevant dataset variables before encoding and scaling. The original raw CSV remains unchanged.

In [44]:
df_clean = df_clean.drop(columns=["rate_max_min_ratio"], errors="ignore")

print("Final cleaned dataset shape:", df_clean.shape)
display(df_clean.head())

Final cleaned dataset shape: (62252, 6)


,id,currency,ratedate,buyingrate,centralrate,sellingrate
0,64143,CFA,2026-10-09,2.2508,2.2608,2.2708
1,64144,YUAN/RENMINBI,2026-10-09,198.7637,198.8384,198.9131
2,64145,DANISH KRONA,2026-10-09,199.2639,199.3388,199.4137
3,64146,EURO,2026-10-09,1489.6753,1490.2353,1490.7952
4,64147,YEN,2026-10-09,8.4024,8.4056,8.4087


## Further Investigation of Extreme Exchange-Rate Values

Although the initial cleaning process included an outlier-screening rule based on the ratio between the highest and lowest exchange rates, some extreme values may not be detected by this rule. Therefore, the records with the highest central rates will be inspected to identify potentially abnormal values before proceeding with categorical encoding and numerical standardization.

In [45]:
# Inspect the records with the highest original central rates
display(
    df_clean.nlargest(
        10, "centralrate"
    )[
        [
            "id",
            "currency",
            "ratedate",
            "centralrate",
            "buyingrate",
            "sellingrate"
        ]
    ]
)

,id,currency,ratedate,centralrate,buyingrate,sellingrate
59554,6723,POUNDS STERLING,2003-10-30,2.248433e+06,2.239917e+06,2.256948e+06
49965,2748,DANISH KRONA,2008-11-14,1.980240e+05,1.971710e+05,1.988770e+05
6048,57001,SDR,2024-10-15,2.287135e+03,2.286444e+03,2.287825e+03
6206,56848,SDR,2024-09-25,2.253404e+03,2.252728e+03,2.254080e+03
6158,56896,SDR,2024-10-02,2.243746e+03,2.243070e+03,2.244423e+03
5858,57199,SDR,2024-11-06,2.241996e+03,2.241329e+03,2.242663e+03
6216,56834,SDR,2024-09-24,2.241936e+03,2.241261e+03,2.242612e+03
6198,56840,POUNDS STERLING,2024-09-25,2.235237e+03,2.234566e+03,2.235907e+03
6146,56908,SDR,2024-10-03,2.234784e+03,2.234110e+03,2.235458e+03
6276,56764,SDR,2024-09-17,2.232997e+03,2.232321e+03,2.233673e+03


In [46]:
# Inspect the record and nearby dates for the same currency
record_id = 6723

record = df_clean.loc[df_clean["id"] == record_id]
currency_name = record["currency"].iloc[0]
record_date = record["ratedate"].iloc[0]

nearby_records = df_clean.loc[
    (df_clean["currency"] == currency_name)
    & (
        df_clean["ratedate"].between(
            record_date - pd.Timedelta(days=14),
            record_date + pd.Timedelta(days=14)
        )
    ),
    [
        "id", "currency", "ratedate",
        "buyingrate", "centralrate", "sellingrate"
    ]
].sort_values("ratedate")

display(nearby_records)

,id,currency,ratedate,buyingrate,centralrate,sellingrate
59614,6713,POUNDS STERLING,2003-10-16,2.146269e+02,2.154630e+02,2.162991e+02
59603,6714,POUNDS STERLING,2003-10-17,2.146397e+02,2.154759e+02,2.163120e+02
59602,6715,POUNDS STERLING,2003-10-20,2.157437e+02,2.165818e+02,2.174199e+02
59591,6716,POUNDS STERLING,2003-10-21,2.151388e+02,2.159745e+02,2.168103e+02
59590,6717,POUNDS STERLING,2003-10-22,2.170948e+02,2.179349e+02,2.187751e+02
59579,6718,POUNDS STERLING,2003-10-23,2.184384e+02,2.192838e+02,2.201291e+02
59578,6719,POUNDS STERLING,2003-10-24,2.193041e+02,2.201528e+02,2.210015e+02
59567,6720,POUNDS STERLING,2003-10-27,2.206010e+02,2.214494e+02,2.222978e+02
59566,6721,POUNDS STERLING,2003-10-28,2.198209e+02,2.206663e+02,2.215117e+02
59555,6722,POUNDS STERLING,2003-10-29,2.237287e+02,2.245792e+02,2.254298e+02


In [47]:
# Remove the confirmed extreme exchange-rate anomaly
"""
Record ID 6723, dated 2003-10-30, contains an extreme Pounds Sterling
exchange-rate spike compared with neighbouring dates.
The rates are far outside the surrounding range, indicating a likely data error.
The record is excluded while the original raw dataset remains unchanged.
"""
rows_before = len(df_clean)

df_clean = df_clean.loc[df_clean["id"] != 6723].copy()

print("Records removed:", rows_before - len(df_clean))
print("Cleaned dataset shape:", df_clean.shape)

Records removed: 1
Cleaned dataset shape: (62251, 6)


In [48]:
# Inspect Danish Krona rates around the suspected 2008 anomaly
nearby_records = df_clean.loc[
    (df_clean["currency"] == "DANISH KRONA")
    & (df_clean["ratedate"].between("2008-11-07", "2008-11-21")),
    [
        "id",
        "currency",
        "ratedate",
        "buyingrate",
        "centralrate",
        "sellingrate"
    ]
].sort_values("ratedate")

display(nearby_records)

,id,currency,ratedate,buyingrate,centralrate,sellingrate
50020,2743,DANISH KRONA,2008-11-07,19.8152,19.9009,19.9866
50005,2744,DANISH KRONA,2008-11-10,20.0354,20.1221,20.2087
50000,2745,DANISH KRONA,2008-11-11,19.7424,19.8278,19.9132
49985,2746,DANISH KRONA,2008-11-12,19.5173,19.6018,19.6862
49980,2747,DANISH KRONA,2008-11-13,19.4212,19.5053,19.5893
49965,2748,DANISH KRONA,2008-11-14,197171.0000,198024.0000,198877.0000
49960,2749,DANISH KRONA,2008-11-17,19.6124,19.6973,19.7821
49945,2750,DANISH KRONA,2008-11-18,19.6018,19.6866,19.7714
49940,2751,DANISH KRONA,2008-11-19,19.5822,19.6669,19.7516
49925,2752,DANISH KRONA,2008-11-20,19.4235,19.5076,19.5916


In [49]:
"""
Remove the confirmed Danish Krona exchange-rate anomaly.
Record ID 2748, dated 2008-11-14, has a central rate of 198,024,
while neighbouring observations are approximately 19-20
The isolated spike across all three rates indicates a likely data error.
The original raw dataset remains unchanged.
"""

rows_before = len(df_clean)

df_clean = df_clean.loc[df_clean["id"] != 2748].copy()

print("Records removed:", rows_before - len(df_clean))
print("Cleaned dataset shape:", df_clean.shape)

Records removed: 1
Cleaned dataset shape: (62250, 6)


## Final Validation

The cleaned dataset is reviewed to confirm that the previous cleaning decisions have been applied consistently. This validation checks the dataset’s dimensions, duplicate records, missing values, data types, and remaining exchange-rate quality flags.

The review provides a final checkpoint before categorical encoding and numerical scaling. It does not automatically remove additional records; any remaining rate-order violations are retained as documented potential anomalies.

Expected outcome: The dataset contains 62,250 records, with no missing values, no non-positive exchange rates, no records meeting the extreme rate-difference rule, and 61 records still flagged for rate-order violations.

In [50]:
rate_columns = ["buyingrate", "centralrate", "sellingrate"]

rate_min = df_clean[rate_columns].min(axis=1)
rate_max = df_clean[rate_columns].max(axis=1)

final_checks = {
    "Rows": len(df_clean),
    "Columns": len(df_clean.columns),
    "Exact duplicate records": df_clean.duplicated(
        subset=[
            "currency", "ratedate",
            "buyingrate", "centralrate", "sellingrate"
        ]
    ).sum(),
    "Missing values": df_clean.isnull().sum().sum(),
    "Invalid dates": df_clean["ratedate"].isnull().sum(),
    "Non-positive rate records": (
        df_clean[rate_columns] <= 0
    ).any(axis=1).sum(),
    "Extreme rate-difference records": (
        rate_max / rate_min.where(rate_min > 0)
    ).gt(5).sum(),
    "Rate-order violations": (
        (df_clean["buyingrate"] > df_clean["centralrate"])
        | (df_clean["centralrate"] > df_clean["sellingrate"])
    ).sum()
}

display(pd.Series(final_checks, name="Result").to_frame())

print("\nData types:")
display(df_clean.dtypes.rename("data_type").to_frame())


,Result
Rows,62250
Columns,6
Exact duplicate records,0
Missing values,0
Invalid dates,0
Non-positive rate records,0
Extreme rate-difference records,0
Rate-order violations,61



Data types:


,data_type
id,int64
currency,str
ratedate,datetime64[us]
buyingrate,float64
centralrate,float64
sellingrate,float64


## Data Cleaning Stage Completion

The final validation confirms that the cleaned dataset contains 62,250 records and six columns. No exact duplicate records, missing values, invalid dates, non-positive exchange-rate records, or records meeting the extreme rate-difference screening rule remain.

A further investigation was conducted because the initial rate-ratio screening did not identify every potentially anomalous observation. Record ID 6723, dated 30 October 2003, was identified as an extreme Pounds Sterling outlier compared with neighbouring observations. Its exchange rates were approximately 10,000 times larger than the surrounding values, so the record was removed from the cleaned dataset.

Similarly, record ID 2748, dated 14 November 2008, was identified as an extreme Danish Krona outlier. Its buying, central, and selling rates were 197,171, 198,024, and 198,877, respectively, while neighbouring observations were approximately 19–20. This isolated spike across all three rates indicated a likely data error, so the record was also removed.

A total of 61 records still violate the expected buying–central–selling rate order. These records are retained because a rate-order violation alone does not establish that an observation is incorrect.

The temporary rate_max_min_ratio diagnostic column was removed because it was used only during the outlier investigation and is not required in the final dataset.

The cleaned dataset is ready for the categorical encoding and numerical standardization required for Level 1 Task 2. The original raw CSV remains unchanged, and the cleaned dataset is maintained separately for reproducibility and future modelling.

## Encoding Categorical Variables

The currency column is a nominal categorical variable with no inherent order. One-hot encoding converts its categories into separate binary columns, with 1 indicating the presence of a currency and 0 otherwise.

The transformation is performed on a separate copy of df_clean to preserve the cleaned dataset for future modelling.

The id column is retained as a record identifier and is neither encoded nor scaled because its numerical values do not represent a meaningful measurement. The ratedate column is also retained in datetime format without encoding so that its temporal information remains available for potential time-series analysis.

The three exchange-rate columns remain numerical and are standardized in the next preprocessing step. The binary currency columns are not scaled.

In [51]:
# Create a separate DataFrame for preprocessing
df_preprocessed = df_clean.copy()

# One-hot encode the currency column
df_preprocessed = pd.get_dummies(
    df_preprocessed,
    columns=["currency"],
    dtype=int
)

print("Original cleaned dataset shape:", df_clean.shape)
print("Encoded dataset shape:", df_preprocessed.shape)
display(df_preprocessed.head())

Original cleaned dataset shape: (62250, 6)
Encoded dataset shape: (62250, 20)


,id,ratedate,buyingrate,centralrate,sellingrate,currency_CFA,currency_DANISH KRONA,currency_EURO,currency_NAIRA,currency_POESO,currency_POUNDS STERLING,currency_RIYAL,currency_SDR,currency_SOUTH AFRICAN RAND,currency_SWISS FRANC,currency_UAE DIRHAM,currency_US DOLLAR,currency_WAUA,currency_YEN,currency_YUAN/RENMINBI
0,64143,2026-10-09,2.2508,2.2608,2.2708,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,64144,2026-10-09,198.7637,198.8384,198.9131,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,64145,2026-10-09,199.2639,199.3388,199.4137,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,64146,2026-10-09,1489.6753,1490.2353,1490.7952,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4,64147,2026-10-09,8.4024,8.4056,8.4087,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


## Standardizing Numerical Features

The buyingrate, centralrate, and sellingrate columns contain numerical exchange-rate values with different ranges. Standardization transforms these columns to have a mean approximately equal to zero and a standard deviation approximately equal to one.

The StandardScaler from scikit-learn is used to standardize the three exchange-rate columns in df_preprocessed. The identifier, date, and one-hot encoded currency columns are excluded from scaling. The original df_clean remains unchanged for future modelling.

In [52]:
from sklearn.preprocessing import StandardScaler

# Select the numerical exchange-rate columns
rate_columns = ["buyingrate", "centralrate", "sellingrate"]

# Create and apply the scaler
scaler = StandardScaler()
df_preprocessed[rate_columns] = scaler.fit_transform(
    df_preprocessed[rate_columns]
)

print("Preprocessed dataset shape:", df_preprocessed.shape)
print("\nStandardized exchange-rate summary:")
display(df_preprocessed[rate_columns].agg(["mean", "std", "min", "max"]).round(3))

display(df_preprocessed.head())

Preprocessed dataset shape: (62250, 20)

Standardized exchange-rate summary:


,buyingrate,centralrate,sellingrate
mean,-0.000,-0.000,-0.000
std,1.000,1.000,1.000
min,-0.632,-0.633,-0.634
max,4.568,4.567,4.566


,id,ratedate,buyingrate,centralrate,sellingrate,currency_CFA,currency_DANISH KRONA,currency_EURO,currency_NAIRA,currency_POESO,currency_POUNDS STERLING,currency_RIYAL,currency_SDR,currency_SOUTH AFRICAN RAND,currency_SWISS FRANC,currency_UAE DIRHAM,currency_US DOLLAR,currency_WAUA,currency_YEN,currency_YUAN/RENMINBI
0,64143,2026-10-09,-0.627281,-0.627900,-0.628545,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,64144,2026-10-09,-0.180352,-0.180995,-0.181637,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,64145,2026-10-09,-0.179214,-0.179857,-0.180499,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,64146,2026-10-09,2.755563,2.754900,2.754418,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4,64147,2026-10-09,-0.613290,-0.613930,-0.614596,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


# Verifying the Preprocessing

In [53]:
print("Missing values:", df_preprocessed.isna().sum().sum())
print("Shape:", df_preprocessed.shape)

currency_columns = [
    column for column in df_preprocessed.columns
    if column.startswith("currency_")
]

print(
    "Currency columns contain only 0 and 1:",
    df_preprocessed[currency_columns]
    .isin([0, 1]).all().all()
)

print(
    "Cleaned dataset unchanged:",
    df_clean.shape == (62250, 6)
)

Missing values: 0
Shape: (62250, 20)
Currency columns contain only 0 and 1: True
Cleaned dataset unchanged: True


## Data Cleaning and Preprocessing Completion

The dataset was cleaned by addressing duplicate records, investigating anomalous exchange rates, and removing records according to the documented cleaning rules. No missing values remained in the cleaned dataset, so imputation was unnecessary.

One-hot encoding was applied to the currency column, creating binary columns for the currency categories. The buyingrate, centralrate, and sellingrate columns were standardized using StandardScaler. The identifier and date columns were retained without encoding or scaling.

The preprocessing checks confirmed that the transformed dataset contains 62,250 records and 20 columns, with no missing values and valid binary currency indicators. The original cleaned dataset, df_clean, remains unchanged for future modelling.

The standardized data is intended to demonstrate preprocessing techniques. Any future predictive modelling will use an appropriate train-test split and fit preprocessing transformations on the training data only to prevent data leakage.

In [54]:
# Save the encoded and standardized dataset
df_preprocessed.to_csv(
    "cbn_exchange_rates_preprocessed.csv",
    index=False,
    encoding="utf-8"
)

print("Preprocessed dataset saved successfully.")
print("Saved file: cbn_exchange_rates_preprocessed.csv")

Preprocessed dataset saved successfully.
Saved file: cbn_exchange_rates_preprocessed.csv


In [55]:
df_clean.to_csv(
    "cbn_exchange_rates_cleaned.csv",
    index=False,
    encoding="utf-8"
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
